# FP Analysis — Model CSIC (CSIC 2010) — v5 (LightGBM)

**Objetivo:** Analizar los False Positives del modelo LightGBM v5 (ratio features).

**Pregunta:** ¿Las ratio features cambian los patrones de FP vs v4?

**MLflow:**
- Model Registry: `model-csic`
- Tracking: `http://localhost:5081`

In [11]:
import os
import sys
from pathlib import Path

# Agregar src al path
ROOT = Path.cwd()
while not (ROOT / 'mkdocs.yml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Config — v5 con ratio features
FEATURES_PATH = ROOT / 'data' / 'processed' / 'csic2010' / 'features_v5.parquet'
MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5081')

print(f'ROOT: {ROOT}')
print(f'Features: {FEATURES_PATH}')
print(f'Features existe: {FEATURES_PATH.exists()}')

ROOT: /Users/permotion/Desktop/repositories/PERMOTION/PMT_MLSecOps
Features: /Users/permotion/Desktop/repositories/PERMOTION/PMT_MLSecOps/data/processed/csic2010/features_v5.parquet
Features existe: True


## 1. Cargar datos y re-entrenar modelo

Para hacer FP analysis necesitamos:
1. Cargar features
2. Re-entrenar LightGBM con los mismos hyperparameters
3. Obtener predicciones en test set
4. Filtrar FP

In [12]:
RANDOM_STATE = 42
TEST_SIZE = 0.30
VAL_SIZE = 0.50
CONTINUOUS_FEATURES = ['url_length', 'url_query_length', 'content_length']

# Cargar features
df = pd.read_parquet(FEATURES_PATH)
print(f'Shape: {df.shape}')
print(f'Label dist: {df['label'].value_counts().to_dict()}')

# Prepare features
feature_cols = [c for c in df.columns if c != 'label']
X = df[feature_cols].values.astype(np.float32)
y = df['label'].values

# Split 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE, stratify=y_temp, random_state=RANDOM_STATE
)

# Scale continuous features
scaler = StandardScaler()
continuous_idx = [feature_cols.index(c) for c in CONTINUOUS_FEATURES]
X_train[:, continuous_idx] = scaler.fit_transform(X_train[:, continuous_idx])
X_val[:, continuous_idx] = scaler.transform(X_val[:, continuous_idx])
X_test[:, continuous_idx] = scaler.transform(X_test[:, continuous_idx])

print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')

Shape: (61065, 28)
Label dist: {0: 36000, 1: 25065}
Train: 42745 | Val: 9160 | Test: 9160


In [13]:
# Re-entrenar LightGBM
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight: {scale_pos_weight:.3f}')

model = LGBMClassifier(
    n_estimators=200,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

model.fit(X_train, y_train)
print('LightGBM entrenado')

scale_pos_weight: 1.436
LightGBM entrenado


## 2. Obtener predicciones y filtrar FP

In [14]:
THRESHOLD = 0.2707  # Threshold óptimo de LightGBM v5

# Predicciones en test
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= THRESHOLD).astype(int)

# Índices en el test set
# Como hicimos train_test_split con indices, necesitamos找回 los indices originales
# Pero para el análisis usamos X_test directamente

# Crear DataFrame con resultados
df_test = df.iloc[-len(y_test):].copy() if len(y_test) == len(df) else None

# Mejor: guardar los indices
df_full = pd.read_parquet(FEATURES_PATH)
df_train, df_temp, _, _ = train_test_split(
    df_full, df_full['label'].values,
    test_size=TEST_SIZE, stratify=df_full['label'].values, random_state=RANDOM_STATE
)
df_val, df_test, _, _ = train_test_split(
    df_temp, df_temp['label'].values,
    test_size=VAL_SIZE, stratify=df_temp['label'].values, random_state=RANDOM_STATE
)

df_test = df_test.reset_index(drop=True)
df_test['proba'] = test_proba
df_test['pred'] = test_pred

print(f'Test set: {len(df_test)}')

Test set: 9160


/Users/permotion/Desktop/repositories/PERMOTION/PMT_MLSecOps/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [15]:
# Clasificar predicciones
tp = df_test[(df_test['pred'] == 1) & (df_test['label'] == 1)]
tn = df_test[(df_test['pred'] == 0) & (df_test['label'] == 0)]
fp = df_test[(df_test['pred'] == 1) & (df_test['label'] == 0)]  # PRED=1, ACTUAL=0
fn = df_test[(df_test['pred'] == 0) & (df_test['label'] == 1)]  # PRED=0, ACTUAL=1

print(f'True Positives (TP):  {len(tp)} — Pred=1, Actual=1 ✅ Ataques detectados')
print(f'True Negatives (TN):   {len(tn)} — Pred=0, Actual=0 ✅ Normales detectados')
print(f'False Positives (FP): {len(fp)} — Pred=1, Actual=0 ❌ Normales marcados como ataque')
print(f'False Negatives (FN): {len(fn)} — Pred=0, Actual=1 ❌ Ataques NO detectados')
print()
print(f'Total: {len(df_test)}')
print(f'Precision: {len(tp) / (len(tp) + len(fp)):.4f}')
print(f'Recall: {len(tp) / (len(tp) + len(fn)):.4f}')

True Positives (TP):  3592 — Pred=1, Actual=1 ✅ Ataques detectados
True Negatives (TN):   4458 — Pred=0, Actual=0 ✅ Normales detectados
False Positives (FP): 942 — Pred=1, Actual=0 ❌ Normales marcados como ataque
False Negatives (FN): 168 — Pred=0, Actual=1 ❌ Ataques NO detectados

Total: 9160
Precision: 0.7922
Recall: 0.9553


## 3. Análisis de FP por método HTTP

Ver si los FP están concentrados en GET, POST o PUT.

In [16]:
# Distribución de FP por método
print('=== FP por método HTTP ===')
fp_method = fp['method_is_get'].value_counts()
print(f'FP en GET:  {fp_method.get(1, 0)}')
print(f'FP en POST: {fp_method.get(0, 0)}')

# Comparar con distribución general de test
print()
print('=== Distribución en test set ===')
test_method = df_test['method_is_get'].value_counts()
print(f'GET en test:  {test_method.get(1, 0)}')
print(f'POST en test: {test_method.get(0, 0)}')

# Tasa de FP por método
fp_get = fp_method.get(1, 0)
fp_post = fp_method.get(0, 0)
test_get = test_method.get(1, 0)
test_post = test_method.get(0, 0)

print()
print(f'Tasa FP en GET:  {fp_get / test_get * 100:.1f}%')
print(f'Tasa FP en POST: {fp_post / test_post * 100:.1f}%')

=== FP por método HTTP ===
FP en GET:  458
FP en POST: 484

=== Distribución en test set ===
GET en test:  6429
POST en test: 2731

Tasa FP en GET:  7.1%
Tasa FP en POST: 17.7%


## 4. Análisis de FP por features binarias

Comparar distribución de indicadores entre FP y TN (los que sí están bien).

In [17]:
BINARY_FEATURES = [
    'method_is_get', 'method_is_post', 'method_is_put',
    'url_has_pct27', 'url_has_pct3c', 'url_has_dashdash', 'url_has_script', 'url_has_select',
    'content_has_pct27', 'content_has_pct3c', 'content_has_dashdash', 'content_has_script', 'content_has_select',
]

print('=== Distribución de features binarias ===')
print(f'{"Feature":<25} {"FP (%)":>10} {"TN (%)":>10} {"Diferencia":>12}')
print('-' * 60)

for feat in BINARY_FEATURES:
    fp_rate = (fp[feat].sum() / len(fp) * 100) if len(fp) > 0 else 0
    tn_rate = (tn[feat].sum() / len(tn) * 100) if len(tn) > 0 else 0
    diff = fp_rate - tn_rate
    print(f'{feat:<25} {fp_rate:>10.1f} {tn_rate:>10.1f} {diff:>+12.1f}')

=== Distribución de features binarias ===
Feature                       FP (%)     TN (%)   Diferencia
------------------------------------------------------------
method_is_get                   48.6       84.1        -35.5
method_is_post                  51.4       15.9        +35.5
method_is_put                    0.0        0.0         +0.0
url_has_pct27                    0.5        0.0         +0.5
url_has_pct3c                    0.0        0.0         +0.0
url_has_dashdash                 0.0        0.0         +0.0
url_has_script                   0.0        0.0         +0.0
url_has_select                   0.0        0.0         +0.0
content_has_pct27                0.8        0.0         +0.8
content_has_pct3c                0.0        0.0         +0.0
content_has_dashdash             0.0        0.0         +0.0
content_has_script               0.0        0.0         +0.0
content_has_select               0.0        0.0         +0.0


## 5. Análisis de features continuas en FP

In [18]:
CONTINUOUS_FEATURES = ['url_length', 'url_query_length', 'content_length', 'url_pct_density', 'content_pct_density']

print('=== Estadísticas de features continuas ===')
print(f'{"Feature":<25} {"FP media":>12} {"TN media":>12} {"Diferencia":>12}')
print('-' * 65)

for feat in CONTINUOUS_FEATURES:
    if feat in df_test.columns:
        fp_mean = fp[feat].mean()
        tn_mean = tn[feat].mean()
        diff = fp_mean - tn_mean
        print(f'{feat:<25} {fp_mean:>12.1f} {tn_mean:>12.1f} {diff:>+12.1f}')

=== Estadísticas de features continuas ===
Feature                       FP media     TN media   Diferencia
-----------------------------------------------------------------
url_length                       144.7         63.4        +81.3
url_query_length                  89.5          6.5        +83.0
content_length                    95.6          4.9        +90.7
url_pct_density                    0.0          0.0         +0.0
content_pct_density                0.0          0.0         +0.0


## 6. Hipótesis: ¿qué está causando los FP?

In [19]:
# Hipótesis 1: ¿Los FP son requests con content_length > 0 (POST) pero son legítimos)?
print('=== Hipótesis 1: FP en POST con content_length > 0 ===')
fp_post_with_content = fp[(fp['method_is_post'] == 0) & (fp['content_length'] > 0)]
print(f'FP que son POST con content_length > 0: {len(fp_post_with_content)}')

# Hipótesis 2: ¿Los FP tienen indicadores de encoding pero son legítimos?
print()
print('=== Hipótesis 2: FP con url_has_pct27=1 (comillas en URL) ===')
fp_pct27 = fp[fp['url_has_pct27'] == 1]
print(f'FP con %27 en URL: {len(fp_pct27)} ({len(fp_pct27)/len(fp)*100:.1f}% de FP)')

# Hipótesis 3: ¿Los FP son requests largas (posiblemente con datos de formulario)?
print()
print('=== Hipótesis 3: FP con content_length > 50 ===')
fp_long_content = fp[fp['content_length'] > 50]
print(f'FP con content_length > 50: {len(fp_long_content)} ({len(fp_long_content)/len(fp)*100:.1f}% de FP)')

# Hipótesis 4: ¿Los FP son GETs con query string larga?
print()
print('=== Hipótesis 4: FP en GET con url_length > 100 ===')
fp_get_long = fp[(fp['method_is_get'] == 1) & (fp['url_length'] > 100)]
print(f'FP GET con url_length > 100: {len(fp_get_long)} ({len(fp_get_long)/len(fp)*100:.1f}% de FP)')

=== Hipótesis 1: FP en POST con content_length > 0 ===
FP que son POST con content_length > 0: 0

=== Hipótesis 2: FP con url_has_pct27=1 (comillas en URL) ===
FP con %27 en URL: 5 (0.5% de FP)

=== Hipótesis 3: FP con content_length > 50 ===
FP con content_length > 50: 482 (51.2% de FP)

=== Hipótesis 4: FP en GET con url_length > 100 ===
FP GET con url_length > 100: 455 (48.3% de FP)


## 7. Próximos pasos según hallazgos

Dependiendo de qué patrones encontremos:

In [20]:
print('=== Resumen de hipótesis ===')
print()
print('Si la mayoría de FP son POST legítimos:')
print('  → Feature: content_length alta en POST normal')
print('  → Solución: Agregar ratio content_length / url_query_length')
print()
print('Si la mayoría de FP tienen %27 en URL:')
print('  → Feature: indicator de comillas en productos/nombres')
print('  → Solución: Crear feature url_has_multiple_pct27')
print()
print('Si la mayoría de FP son GETs largos:')
print('  → Feature: url_length combinado con method')
print('  → Solución: url_length * method_is_get')

=== Resumen de hipótesis ===

Si la mayoría de FP son POST legítimos:
  → Feature: content_length alta en POST normal
  → Solución: Agregar ratio content_length / url_query_length

Si la mayoría de FP tienen %27 en URL:
  → Feature: indicator de comillas en productos/nombres
  → Solución: Crear feature url_has_multiple_pct27

Si la mayoría de FP son GETs largos:
  → Feature: url_length combinado con method
  → Solución: url_length * method_is_get
